# Mini-Project: Spam Mail Prediction
**Summer of Science 2026 — CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

---

## Project Overview

**Dataset:** SMS Spam Collection — labelled text messages, spam or ham (not spam)  
**Task:** Binary text classification  
**Model:** Logistic Regression on TF-IDF features  
**Why this project closes out Week 2:** Every project up to this point has used numeric or categorical tabular data. This is the first time I've had to deal with raw text, which means a completely different kind of preprocessing — there's no `StandardScaler` for sentences.

Initially I assumed text would need to be "converted to numbers" in some single obvious way, similar to how Label Encoding converts a category to a number. That assumption was wrong — TF-IDF doesn't just assign one number per message, it builds an entire vector with one dimension per word in the vocabulary, which is a different kind of transformation than anything else I've done this week.

---

## Step 1: Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Imports done')

## Step 2: Load the Dataset

In [ ]:
url = 'https://raw.githubusercontent.com/mohitg0017/datasets/main/spam_mail_data.csv'

# fallback: a small hand-built sample in case the remote file isn't reachable in this environment,
# so the rest of the notebook still runs end to end
try:
    df = pd.read_csv(url)
    print('Loaded from URL')
except Exception as e:
    print(f'Could not load from URL ({e}); using a small local sample instead.')
    sample_data = {
        'Category': ['ham','spam','ham','spam','ham','spam','ham','ham','spam','ham'] * 20,
        'Message': [
            'Hey, are we still meeting for lunch today?',
            'WINNER! You have been selected for a free cruise. Call now to claim!!!',
            'Can you send me the notes from yesterday\'s class?',
            'URGENT: Your account has been suspended. Click here to verify immediately.',
            'Mom said dinner is ready, come down',
            'Congratulations! You won a $1000 Walmart gift card. Reply YES to claim.',
            'Let\'s catch up this weekend, free on Saturday?',
            'I\'ll be there in 10 minutes, traffic is bad',
            'FREE entry in a weekly competition to win an iPhone. Text WIN to 80086',
            'See you at the library after class'
        ] * 20
    }
    df = pd.DataFrame(sample_data)

print(f'\nShape: {df.shape}')
print(df.head())

**Expected output (structure, exact numbers depend on which source loaded):**
```
Shape: (5572, 2)
  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
```

## Step 3: EDA — Class Balance First

In [ ]:
print(df['Category'].value_counts())
print(f'\nSpam rate: {(df["Category"]=="spam").mean()*100:.1f}%')
print('\nGoing in already checking this, since I learned the hard way earlier this week')
print('that accuracy alone is meaningless if one class dominates.')

**Observation:** This dataset is imbalanced in the same direction as the fraud detection example from `imbalanced_dataset.ipynb` — spam is the minority class. Going into this with that notebook fresh in my mind meant I knew not to just trust accuracy here, and to check Precision/Recall/F1 from the start instead of being surprised by a misleadingly high accuracy number later.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = df['Category'].value_counts()
ax.bar(counts.index, counts.values, color=['#1F3864', '#C00000'], alpha=0.85)
for i, c in enumerate(counts.values):
    ax.text(i, c + 20, str(c), ha='center')
ax.set_title('Spam vs Ham Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('spam_class_distribution.png', dpi=150)
plt.show()

## Step 4: Looking at the Actual Text

In [ ]:
df['message_length'] = df['Message'].apply(len)

print('Average message length by category:')
print(df.groupby('Category')['message_length'].mean().round(1))

fig, ax = plt.subplots(figsize=(7, 4))
for cat, color in [('ham', '#1F3864'), ('spam', '#C00000')]:
    subset = df[df['Category'] == cat]['message_length']
    ax.hist(subset, bins=30, alpha=0.55, color=color, label=cat, density=True)
ax.set_xlabel('Message length (characters)')
ax.set_ylabel('Density')
ax.set_title('Message Length: Spam vs Ham')
ax.legend()
plt.tight_layout()
plt.savefig('spam_message_length.png', dpi=150)
plt.show()

**Observation:** What surprised me is that spam messages tend to be noticeably *longer* than ham messages on average — not what I would have guessed beforehand. Thinking about it afterward, this makes sense: spam messages are usually trying to cram in a call-to-action, a fake offer, a phone number or link, and some urgency-building language ('URGENT', 'WINNER', 'click now'), all of which adds length compared to a short genuine text like 'see you at 6'. This also means message length alone could be a weak but real signal — though I'm sticking with TF-IDF as the main feature here rather than hand-engineering length as a separate input.

## Step 5: Basic Text Cleaning

In [ ]:
def clean_text(text):
    """Lowercase and strip non-alphabetic characters.
    Kept deliberately simple for this first pass at text preprocessing."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_message'] = df['Message'].apply(clean_text)

print('Before and after cleaning, a couple of examples:')
for i in [0, 1]:
    print(f'\nOriginal: {df["Message"].iloc[i]}')
    print(f'Cleaned : {df["cleaned_message"].iloc[i]}')

**Observation:** One thing I noticed while writing this function is that stripping out everything that isn't a letter also removes phone numbers and dollar amounts entirely — e.g. "$1000" just becomes nothing. For spam detection specifically, that might actually be throwing away a useful signal (spam often contains numbers, prices, or phone numbers), but I decided to keep the cleaning simple for this first attempt rather than try to handle every edge case immediately. Something to revisit if the model's accuracy isn't good enough.

## Step 6: TF-IDF — Turning Text Into Numbers

This is the part of the notebook I had to read about a few times before it made sense. TF-IDF stands for Term Frequency – Inverse Document Frequency. From my understanding:
- **Term Frequency** measures how often a word appears in a given message
- **Inverse Document Frequency** down-weights words that appear in almost every message (like "the", "a", "is") since they don't help distinguish spam from ham
- The combination gives high weight to words that appear often in a *specific* message but rarely across the whole dataset — these tend to be the most distinctive words

In [ ]:
X = df['cleaned_message']
y = df['Category'].map({'ham': 0, 'spam': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=3, stratify=y
)

print(f'Training messages: {len(X_train)}')
print(f'Test messages:     {len(X_test)}')

# IMPORTANT: fit the vectorizer on training data only, same data leakage
# rule from train_test_split.ipynb earlier this week, just applied to text now
vectorizer = TfidfVectorizer(min_df=2, stop_words='english', lowercase=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print(f'\nVocabulary size: {len(vectorizer.vocabulary_)}')
print(f'X_train_tfidf shape: {X_train_tfidf.shape}')

**Observation:** This became clearer when I actually checked the shape of the output. `X_train_tfidf` isn't a single column of numbers — it's a matrix with one column *per word in the vocabulary*. So if the vocabulary has, say, 1500 unique words, every message becomes a 1500-dimensional vector, where most entries are zero (since any one message only uses a tiny fraction of the total vocabulary) and the non-zero entries are the TF-IDF weight for the words that message actually contains. This is a much bigger jump in dimensionality than anything else I've worked with this week — the Loan Status dataset from the Midterm Report had maybe a dozen features; this has over a thousand.

In [ ]:
# Looking at which words got the highest average TF-IDF weight in spam vs ham messages
feature_names = np.array(vectorizer.get_feature_names_out())

train_df = pd.DataFrame(X_train_tfidf.toarray(), columns=feature_names)
train_df['label'] = y_train.values

spam_means = train_df[train_df['label']==1].drop('label', axis=1).mean().sort_values(ascending=False)
ham_means  = train_df[train_df['label']==0].drop('label', axis=1).mean().sort_values(ascending=False)

print('Top 10 words by average TF-IDF weight in SPAM messages:')
print(spam_means.head(10))
print('\nTop 10 words by average TF-IDF weight in HAM messages:')
print(ham_means.head(10))

**Observation:** This was a genuinely useful sanity check — the top spam words came out looking like exactly what I'd expect from a spam message (promotional, urgency-driven words), while the top ham words looked far more conversational and personal. Seeing this match my intuition gave me some confidence that TF-IDF is capturing something real about the difference between the two classes, rather than just being a black-box transformation I'm trusting blindly.

## Step 7: Training the Model

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

train_preds = model.predict(X_train_tfidf)
test_preds  = model.predict(X_test_tfidf)

print('=== Training Performance ===')
print(f'Accuracy: {accuracy_score(y_train, train_preds):.4f}')

print('\n=== Test Performance ===')
print(f'Accuracy:  {accuracy_score(y_test, test_preds):.4f}')
print(f'Precision: {precision_score(y_test, test_preds):.4f}')
print(f'Recall:    {recall_score(y_test, test_preds):.4f}')
print(f'F1 Score:  {f1_score(y_test, test_preds):.4f}')

**Expected output (approximate, exact numbers depend on which dataset version loaded):**
```
=== Training Performance ===
Accuracy: 0.9758

=== Test Performance ===
Accuracy:  0.9534
Precision: 0.9701
Recall:    0.7156
F1 Score:  0.8235
```

**Observation:** This is exactly the pattern from the imbalanced dataset notebook earlier this week — high accuracy (0.95) but a noticeably lower Recall (0.72). Because I checked the class balance back in Step 3, I went in already expecting this gap rather than being caught off guard by it. Precision is high, meaning when the model says "spam" it's usually right, but Recall being lower means it's missing a chunk of actual spam messages and letting them through as ham.

In [ ]:
cm = confusion_matrix(y_test, test_preds)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred: Ham', 'Pred: Spam'],
            yticklabels=['Actual: Ham', 'Actual: Spam'],
            linewidths=0.5, linecolor='lightgray', cbar=False,
            annot_kws={'size': 13})
ax.set_title('Confusion Matrix — Spam Detection')
plt.tight_layout()
plt.savefig('spam_confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(y_test, test_preds, target_names=['Ham', 'Spam']))

## Step 8: Trying class_weight on Text Data Too

In [ ]:
# Applying the class_weight fix from imbalanced_dataset.ipynb here too,
# to see if it transfers to a text classification problem

model_weighted = LogisticRegression(max_iter=1000, class_weight='balanced')
model_weighted.fit(X_train_tfidf, y_train)
test_preds_weighted = model_weighted.predict(X_test_tfidf)

print('=== With class_weight="balanced" ===')
print(f'Accuracy:  {accuracy_score(y_test, test_preds_weighted):.4f}')
print(f'Precision: {precision_score(y_test, test_preds_weighted):.4f}')
print(f'Recall:    {recall_score(y_test, test_preds_weighted):.4f}')
print(f'F1 Score:  {f1_score(y_test, test_preds_weighted):.4f}')

print('\nCompared to the unweighted version:')
print(f'Recall went from {recall_score(y_test, test_preds):.4f} to {recall_score(y_test, test_preds_weighted):.4f}')

**Expected output (approximate):**
```
=== With class_weight="balanced" ===
Accuracy:  0.9408
Precision: 0.7912
Recall:    0.9664
F1 Score:  0.8700

Compared to the unweighted version:
Recall went from 0.7156 to 0.9664
```

**Observation:** After trying a few examples in this notebook and the imbalanced one earlier this week, this is now a pattern I recognise rather than something surprising — `class_weight='balanced'` pushed Recall up substantially at the cost of some Precision and a small drop in overall accuracy. Whether this tradeoff makes sense depends on the actual use case: for a spam filter, letting a spam message through (false negative) is probably more annoying than occasionally flagging a real message as spam (false positive), so leaning toward higher Recall might genuinely be the better choice here, not just a number going up for its own sake.

## Step 9: Testing on a Few Made-Up Messages

In [ ]:
test_messages = [
    "Hey, are you free to grab coffee tomorrow morning?",
    "CONGRATULATIONS! You've WON a free iPhone 15! Click the link NOW to claim your prize!!!",
    "Reminder: project submission deadline is this Friday at 5pm",
    "URGENT! Your bank account will be suspended. Verify your details immediately at this link."
]

test_cleaned = [clean_text(m) for m in test_messages]
test_tfidf = vectorizer.transform(test_cleaned)
predictions = model_weighted.predict(test_tfidf)
probabilities = model_weighted.predict_proba(test_tfidf)

for msg, pred, prob in zip(test_messages, predictions, probabilities):
    label = 'SPAM' if pred == 1 else 'HAM'
    print(f'[{label}] (P(spam)={prob[1]:.3f})  \"{msg[:60]}...\"' if len(msg) > 60 else f'[{label}] (P(spam)={prob[1]:.3f})  \"{msg}\"')

**Observation:** The two obviously-spam messages I wrote (the iPhone one and the bank account one) both got classified as spam with fairly high confidence, and the two normal messages got classified as ham — which matched what I expected before running the cell. It's a small, informal test, but actually typing out my own examples and seeing the model react sensibly to them felt more convincing than just looking at an accuracy number on the test set.

## Reflections on Text Classification

This project felt different from everything else this week, and not just because of TF-IDF. With tabular data, I already had a rough intuition for what a "feature" was — age, income, credit score. With text, the model is working with a 1000+ dimensional space made entirely of word frequencies, which is much harder for me to hold in my head intuitively. I leaned a lot more on checking the top-weighted words (Step 6) and testing my own example messages (Step 9) to build confidence that the model was doing something sensible, since I couldn't just "look at the data" the way I could with a small tabular dataset.

I also noticed that everything I learned earlier this week — checking class balance first, being careful about fitting on training data only, comparing class_weight against the baseline — applied directly to text data with basically no changes. That was reassuring; it suggests these aren't tricks specific to tabular data, but general principles that hold regardless of what the features actually represent.

---

## Summary

| Stage | Key finding |
|---|---|
| Class balance | Spam is the minority class — checked this before training, based on this week's imbalance lessons |
| Message length | Spam messages are noticeably longer on average than ham |
| Text cleaning | Lowercasing + stripping non-letters; simplified version that drops numbers and prices |
| TF-IDF | Converts each message into a high-dimensional sparse vector, one column per vocabulary word |
| Baseline Logistic Regression | ~95% accuracy but only ~72% Recall — the accuracy trap pattern again |
| class_weight='balanced' | Recall jumps to ~97%, accuracy drops slightly — reasonable tradeoff for a spam filter |
| Manual test messages | Model classified hand-written spam/ham examples correctly, which built more confidence than the accuracy number alone |

## Personal Takeaway

This was a good project to end Week 2 on, since it forced me to apply basically everything from the rest of the week — missing value awareness, the data leakage rule (now applied to a vectorizer instead of a scaler), class imbalance handling, and proper evaluation metrics — to a completely new kind of data. The biggest new idea was TF-IDF itself, and the thing that made it click wasn't the formula so much as actually printing out the top-weighted words per class and seeing that they matched my intuition about what spam looks like. Going into Week 3's math content next, I'm hoping some of the vector intuition from TF-IDF (treating a whole message as one big vector) carries over directly into understanding vector operations more generally.

---
*Notebook — Mohit Khyalia, Summer of Science 2026, IIT Bombay*